[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PacktPublishing/Data-Strategy-for-LLMs/blob/main/chapter_09/Jupyter_Notebooks/Chapter_9_Notebook.ipynb)

**Click the badge above to run this notebook in Google Colab (no local setup needed).**


# Chapter 9: Task-Specific Evaluation Data

This notebook evaluates the full pipeline built across the book:
1. **Offline evaluation (Part 0)** -- build a golden dataset, compare prompts, score with an LLM judge
2. **RAG system (Part 1, Chapters 4-5)** -- does retrieval find the right documents? Does generation stay faithful?
3. **Multi-metric evaluation (Part 1b)** -- apply Siddall's eight-attribute rubric, see what accuracy alone misses
4. **Synthetic data (Part 2+)** -- are the QA pairs faithful and diverse?
5. **Fine-tuned models** -- does the fine-tuned model beat the base?

We start with a simple offline prompt comparison (Part 0) to learn how LLM-as-judge scoring works,
then apply the same technique to evaluate the RAG pipeline (Part 1), then show why you need
more than one metric (Part 1b), then evaluate each chapter's outputs in turn.

**Run the Shared Setup cell first.**

## Setup

**IMPORTANT: This chapter uses the book-wide shared environment. Follow the README.md in the repository root.**

**Prerequisites: Run Chapter 4, Chapter 6, and Chapter 7 notebooks first.**
- **Chapter 4** creates the shared ChromaDB vector store (`data/chroma_db/`) used for RAG evaluation in Part 0.
- **Chapter 6** produces the synthetic datasets (QA pairs, preference pairs) evaluated in Parts 2-5.
- **Chapter 7** produces a fine-tuned SFT model via the OpenAI API and a LoRA adapter saved locally, evaluated in Parts 6-9.

If you skip Chapter 7, the notebook will fall back to the base model for comparison, but you will miss the fine-tuned vs base comparison which is the whole point.

**Before running:**

1. Run the book-wide setup once from the repository root:
   - macOS/Linux: `bash setup/setup_mac.sh`
   - Windows (PowerShell): `powershell -ExecutionPolicy Bypass -File setup/setup_windows.ps1`

   This creates the `data_strategy_env` environment, registers the **"Python (Data Strategy Book)"** Jupyter kernel, and configures your API key.

2. Select the **"Python (Data Strategy Book)"** kernel (top-right). If it is not listed: Command Palette -> "Developer: Reload Window".

3. Run the **Chapter 4 notebook** to create the shared ChromaDB at `data/chroma_db/`.

4. Run the **Chapter 7 notebook** to produce:
   - `chapter_07/datasets/sft_model_id.txt` (the fine-tuned model ID from OpenAI)
   - `chapter_07/datasets/lora_adapter/` (the saved LoRA adapter weights)

   The fine-tuned model is tied to your OpenAI account. You need the same API key that created it.

**Google Colab users:**
- The LoRA adapter (~1.2 MB) is already committed in the repo, so Colab loads it automatically when you clone. No retraining needed.
- The SFT model ID file contains the author's model ID, which only works with the author's API key. To get your own fine-tuned model, run the Chapter 7 notebook first with your own API key. The notebook will save your model ID to `sft_model_id.txt`.
- If you train the LoRA adapter on Colab, the files save to Colab's ephemeral filesystem and disappear when the session end. To keep them, mount Google Drive first or download the `lora_adapter/` folder before disconnecting.

The next cell installs any missing packages **into the running kernel** and loads your API key (it prompts you if no `.env` key is found, e.g., on Colab). Then run all cells.

In [1]:
import warnings; warnings.filterwarnings("ignore")
# === Chapter 9 Setup: kernel-aware install + API key ===
# Works on local ("Python (Data Strategy Book)" kernel), Google Colab, and fresh environments.
import sys, subprocess
from pathlib import Path

def _install(pkg):
    """Install into the RUNNING kernel's Python (not the shell pip), with fallbacks."""
    for cmd in (
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        [sys.executable, "-m", "pip", "install", pkg, "--user", "--quiet"],
        [sys.executable, "-m", "pip", "install", pkg, "--break-system-packages", "--quiet"],
    ):
        try:
            subprocess.run(cmd, check=True, capture_output=True, text=True)
            return True
        except subprocess.CalledProcessError:
            continue
    return False

for _pkg in ("openai", "pandas", "numpy", "python-dotenv"):
    if not _install(_pkg):
        print(f"WARNING: could not install {_pkg} (restart the kernel and re-run this cell)")

import os, json
import pandas as pd
import numpy as np
from openai import OpenAI
from collections import Counter
from datetime import datetime

# --- Load the OpenAI API key the same way as the rest of the book (utils/config.py) ---
repo_root = Path.cwd()
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / "utils" / "config.py").exists():
        repo_root = _p
        break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from utils.config import get_openai_api_key
    api_key = get_openai_api_key()          # searches up for .env, raises a helpful error if missing
except Exception:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        import getpass
        api_key = getpass.getpass("Enter your OpenAI API key: ")   # Colab / no-.env fallback
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=api_key)

# --- Self-updating model discovery (shared across all chapters) ---
from utils.models import get_best_available_model, patch_client_compat
patch_client_compat(client)   # tolerate newer-model parameter rules (temperature, max_tokens)
BASE_MODEL = get_best_available_model(client)
JUDGE_MODEL = BASE_MODEL
print(f"Judge model: {JUDGE_MODEL}")
print("Setup complete.")

Selected model: gpt-5.5  (live, 69 candidates)
Judge model: gpt-5.5
Setup complete.


## Part 0: Offline Evaluation -- Your Unit Test Suite

Before we evaluate the RAG system or fine-tuned models, this section shows what offline evaluation looks like in practice. We build a small golden dataset (5 questions with known correct answers), run a model against it, score each response with an LLM judge, then change the prompt and watch scores change.

This is exactly the regression detection loop described in the chapter: change something, run the suite, see what broke.

In [2]:
# === Step 1: Build a Golden Dataset ===
# Fixed test cases with known correct answers. This is your unit test suite.
# Tag each one with category and difficulty so you can slice results later.

golden_dataset = [
    {
        "query": "What is the refund policy for digital products?",
        "reference_answer": "Digital products can be refunded within 14 days if not downloaded.",
        "category": "policy",
        "difficulty": "easy"
    },
    {
        "query": "Can I return a physical item after 30 days?",
        "reference_answer": "Physical items must be returned within 30 days with original packaging.",
        "category": "policy",
        "difficulty": "easy"
    },
    {
        "query": "What happens if my subscription renews and I want to cancel?",
        "reference_answer": "You can cancel within 48 hours of renewal for a full refund. After that, the subscription runs until the end of the billing period.",
        "category": "policy",
        "difficulty": "medium"
    },
    {
        "query": "I bought a gift card and the recipient lost it. Can I get a replacement?",
        "reference_answer": "Lost gift cards cannot be replaced unless you have the original receipt and card number.",
        "category": "edge_case",
        "difficulty": "hard"
    },
    {
        "query": "My order arrived damaged. Who pays for return shipping?",
        "reference_answer": "For damaged items, we provide a prepaid return label at no cost to you.",
        "category": "edge_case",
        "difficulty": "medium"
    }
]

print(f'Golden dataset: {len(golden_dataset)} test cases')
print(f'Categories: {set(g["category"] for g in golden_dataset)}')
print(f'Difficulties: {set(g["difficulty"] for g in golden_dataset)}')

Golden dataset: 5 test cases
Categories: {'edge_case', 'policy'}
Difficulties: {'hard', 'easy', 'medium'}


In [3]:
# === Step 2: Define Two Prompts to Compare ===
# Prompt A gives the model the reference answer as policy context.
# Prompt B gives the model nothing, just the question.
# Same model, same judge, same questions. The only variable is the prompt.

PROMPT_A = """You are a customer support assistant. Answer the question using ONLY the provided policy.
If the answer is not in the policy, say "I don't have that information."

Policy: {reference}

Question: {query}"""

PROMPT_B = """Answer this customer question.

Question: {query}"""

print("Prompt A: with policy context (simulates RAG)")
print("Prompt B: no context (baseline)")
print("Same 5 questions, same model, same judge. Only the prompt changes.")

Prompt A: with policy context (simulates RAG)
Prompt B: no context (baseline)
Same 5 questions, same model, same judge. Only the prompt changes.


In [4]:
# === Step 3: Run Both Prompts Through the Model and Score ===

def run_offline_eval(prompt_template, golden_data, label):
    """Run one offline evaluation pass. Returns scores per example."""
    results = []
    for item in golden_data:
        # Build prompt
        if '{reference}' in prompt_template:
            prompt = prompt_template.format(query=item['query'], reference=item['reference_answer'])
        else:
            prompt = prompt_template.format(query=item['query'])

        # Get model response
        resp = client.chat.completions.create(
            model=BASE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        model_answer = resp.choices[0].message.content.strip()
        print(f'[{label}] Query: {item["query"]} | \n Model answer: {model_answer}')

        # Score with LLM judge
        judge_prompt = f"""Score this response against the reference on a 1-5 scale.
1 = completely wrong, 5 = matches reference perfectly.

Question: {item['query']}
Reference: {item['reference_answer']}
Response: {model_answer}

Return ONLY a JSON object: {{"score": N, "reason": "one sentence"}}"""
        print(f'[{label}] Judge prompt: {judge_prompt}')

        judge_resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.0
        )
        content = judge_resp.choices[0].message.content.strip()
        print(f'[{label}] Judge response: {content}')
        if '```json' in content:
            content = content.split('```json')[1].split('```')[0].strip()
        elif '```' in content:
            content = content.split('```')[1].split('```')[0].strip()
        scores = json.loads(content)
        scores['query'] = item['query'][:50]
        scores['category'] = item['category']
        scores['answer_preview'] = model_answer[:80]
        results.append(scores)

    return pd.DataFrame(results)


# Run Prompt A (with context)
print('\n=== Prompt A: With policy context ===')
results_a = run_offline_eval(PROMPT_A, golden_dataset, 'A')
for _, row in results_a.iterrows():
    print(f'  [{row["score"]}/5] {row["query"]}')
    print(f'         {row["reason"]}')

print(f'\n  Average score: {results_a["score"].mean():.2f}/5')

# Run Prompt B (no context -- should score worse)
print('\n=== Prompt B: Without policy context ===')
results_b = run_offline_eval(PROMPT_B, golden_dataset, 'B')
for _, row in results_b.iterrows():
    print(f'  [{row["score"]}/5] {row["query"]}')
    print(f'         {row["reason"]}')

print(f'\n  Average score: {results_b["score"].mean():.2f}/5')


=== Prompt A: With policy context ===
[A] Query: What is the refund policy for digital products? | 
 Model answer: Digital products can be refunded within 14 days if they have not been downloaded.
[A] Judge prompt: Score this response against the reference on a 1-5 scale.
1 = completely wrong, 5 = matches reference perfectly.

Question: What is the refund policy for digital products?
Reference: Digital products can be refunded within 14 days if not downloaded.
Response: Digital products can be refunded within 14 days if they have not been downloaded.

Return ONLY a JSON object: {"score": N, "reason": "one sentence"}
[A] Judge response: {"score":5,"reason":"The response exactly matches the reference policy, stating that digital products are refundable within 14 days if not downloaded."}
[A] Query: Can I return a physical item after 30 days? | 
 Model answer: No. Physical items must be returned within 30 days with original packaging.
[A] Judge prompt: Score this response against the ref

In [5]:
# === Step 4: Compare Results Side by Side ===
# The per-question breakdown shows exactly where things broke.

print('=' * 60)
print('PROMPT COMPARISON -- SIDE BY SIDE')
print('=' * 60)

comparison = results_a[['query', 'score']].rename(columns={'score': 'Prompt_A'}).copy()
comparison['Prompt_B'] = results_b['score'].values
comparison['Delta'] = comparison['Prompt_A'] - comparison['Prompt_B']

for _, row in comparison.iterrows():
    winner = 'A wins' if row['Delta'] > 0 else ('tie' if row['Delta'] == 0 else 'B wins')
    print(f'  A={row["Prompt_A"]}  B={row["Prompt_B"]}  ({winner:6s})  {row["query"]}')

print(f'\n  Average:  A={results_a["score"].mean():.2f}  B={results_b["score"].mean():.2f}')
print(f'  Delta:    {comparison["Delta"].mean():+.2f} (positive = A is better)')
print()
print('Prompt A scored higher on every question. The gift card edge case')
print('was the biggest gap -- the model with no context invented a policy')
print('that does not exist. That is exactly the regression you want to')
print('catch before deployment.')

PROMPT COMPARISON -- SIDE BY SIDE
  A=5  B=3  (A wins)  What is the refund policy for digital products?
  A=5  B=4  (A wins)  Can I return a physical item after 30 days?
  A=5  B=3  (A wins)  What happens if my subscription renews and I want 
  A=5  B=3  (A wins)  I bought a gift card and the recipient lost it. Ca
  A=5  B=5  (tie   )  My order arrived damaged. Who pays for return ship

  Average:  A=5.00  B=3.60
  Delta:    +1.40 (positive = A is better)

Prompt A scored higher on every question. The gift card edge case
was the biggest gap -- the model with no context invented a policy
that does not exist. That is exactly the regression you want to
catch before deployment.


## Part 1: Evaluating the RAG Pipeline (Chapters 4-5)

Now that you know how LLM-as-judge scoring works, we apply the same technique to evaluate the full RAG system built in Chapters 4 and 5.

A RAG pipeline can fail at two independent stages:
1. **Retrieval** -- did the vector store surface the right documents for the query?
2. **Generation** -- did the LLM produce a correct, faithful answer from the retrieved context?

Both stages need their own metrics. We evaluate them here: first with hand-built metrics you can understand line by line, then with RAGAS, an open-source framework that automates the same checks at scale.

**Prerequisites:** Run the Chapter 4 notebook first (creates the shared ChromaDB at `data/chroma_db/`).

In [6]:
# === RAG Step 1: Load the ChromaDB From Chapter 4 ===
# The Chapter 4 notebook created a shared vector store with four company documents.
# We load that same collection here.

import chromadb

SHARED_DB = repo_root / 'data' / 'chroma_db'
if not SHARED_DB.exists():
    raise FileNotFoundError(
        f"ChromaDB not found at {SHARED_DB}. Run the Chapter 4 notebook first."
    )

rag_client = chromadb.PersistentClient(path=str(SHARED_DB))
book_collection = rag_client.get_or_create_collection(name="book_collection")
print(f"Loaded collection: {book_collection.name} ({book_collection.count()} documents)")

# Show what is in the collection (you should recognise these from Chapter 4)
existing = book_collection.get()
for doc_id, doc_text in zip(existing["ids"], existing["documents"]):
    print(f"  [{doc_id}] {doc_text[:80]}...")

Loaded collection: book_collection (24 documents)
  [manual_1] This is a quick test sentence to index....
  [demo_ce405c63b181a6ab] First line...
  [demo_b456a020d1eecda5] Second line...
  [doc_e703d12f46044bda] The company's new AI policy, effective June 1st, requires all employees to compl...
  [doc_3881aea93a79aa81] Our Q2 financial results show a 15% increase in revenue, driven by strong sales ...
  [doc_8a91f8af1bf32f5e] The Aurora Project, our next-generation AI platform, is scheduled for a beta rel...
  [doc_7da6c838a91b28c5] All travel and expense reports must be submitted through the new online portal b...
  [id_0] The company's new AI policy, effective June 1st, requires all employees to compl...
  [id_1] Our Q2 financial results show a 15% increase in revenue, driven by strong sales ...
  [id_2] The Aurora Project, our next-generation AI platform, is scheduled for a beta rel...
  [id_3] All travel and expense reports must be submitted through the new online portal b...
  [ra

In [7]:
# === RAG Step 2: Build the RAG Golden Dataset ===
# Same idea as Part 0, but now each test case includes the document ID we
# expect the retriever to return. This lets us check retrieval accuracy
# separately from generation quality.

rag_golden = [
    {
        "query": "What is the new AI policy?",
        "expected_doc_id": "id_0",
        "reference_answer": "The new AI policy requires all employees to complete a mandatory training course, effective June 1st.",
        "category": "policy",
    },
    {
        "query": "What were the Q2 financial results?",
        "expected_doc_id": "id_1",
        "reference_answer": "Q2 results show a 15% increase in revenue, driven by strong sales in the European market.",
        "category": "finance",
    },
    {
        "query": "When is the Aurora Project beta release?",
        "expected_doc_id": "id_2",
        "reference_answer": "The Aurora Project is scheduled for a beta release in the third quarter.",
        "category": "project",
    },
    {
        "query": "How do I submit expense reports?",
        "expected_doc_id": "id_3",
        "reference_answer": "Expense reports must be submitted through the new online portal by the 25th of each month.",
        "category": "process",
    },
    {
        "query": "What is our remote work policy?",
        "expected_doc_id": None,  # intentionally out of scope
        "reference_answer": "No information available in the knowledge base.",
        "category": "out_of_scope",
    },
]

print(f"RAG golden dataset: {len(rag_golden)} test cases")
print(f"Categories: {sorted(set(g['category'] for g in rag_golden))}")
print(f"Out-of-scope questions: {sum(1 for g in rag_golden if g['expected_doc_id'] is None)}")
print(f"\nThe expected_doc_id lets us check retrieval accuracy separately from generation.")

RAG golden dataset: 5 test cases
Categories: ['finance', 'out_of_scope', 'policy', 'process', 'project']
Out-of-scope questions: 1

The expected_doc_id lets us check retrieval accuracy separately from generation.


In [8]:
# === RAG Step 3: Run the Full Pipeline and Score ===
# For each question: retrieve -> check hit -> generate -> judge on faithfulness + correctness

def evaluate_rag(collection, golden_data, n_results=2):
    """Evaluate retrieval AND generation quality for a RAG pipeline."""
    rows = []
    for item in golden_data:
        # 1. Retrieve
        retrieval = collection.query(
            query_texts=[item["query"]],
            n_results=n_results,
            include=["documents", "distances"],
        )
        retrieved_ids = retrieval["ids"][0]
        retrieved_docs = retrieval["documents"][0]
        distances = retrieval["distances"][0]

        # 2. Check retrieval hit
        expected = item["expected_doc_id"]
        retrieval_hit = expected in retrieved_ids if expected else True
        top_distance = distances[0] if distances else float("inf")

        # 3. Generate with retrieved context
        context = "\n---\n".join(retrieved_docs)
        prompt = (
            "You are an expert assistant. Use ONLY the following context to answer.\n"
            "If the answer is not in the context, say 'I cannot find this information.'\n\n"
            f"<context>{context}</context>\n"
            f"<question>{item['query']}</question>\nAnswer:"
        )
        gen_resp = client.chat.completions.create(
            model=BASE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        answer = gen_resp.choices[0].message.content.strip()

        # 4. Judge: faithfulness + correctness
        judge_prompt = (
            "Score this RAG response on two dimensions (1-5 each).\n"
            "1. Faithfulness: answer uses ONLY retrieved context, no fabrication (5=grounded, 1=hallucinated)\n"
            "2. Correctness: answer matches the reference (5=perfect, 1=wrong)\n\n"
            f"Context: {context[:500]}\n"
            f"Question: {item['query']}\n"
            f"Reference: {item['reference_answer']}\n"
            f"Response: {answer}\n\n"
            'Return ONLY JSON: {"faithfulness": N, "correctness": N, "reason": "one sentence"}'
        )
        judge_resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.0,
        )
        raw = judge_resp.choices[0].message.content.strip()
        if "```json" in raw:
            raw = raw.split("```json")[1].split("```")[0].strip()
        elif "```" in raw:
            raw = raw.split("```")[1].split("```")[0].strip()
        scores = json.loads(raw)

        hit_label = "HIT" if retrieval_hit else "MISS"
        print(
            f"  [{hit_label}] {item['query'][:50]:50s} | "
            f"Faith={scores['faithfulness']} Corr={scores['correctness']} | "
            f"{answer[:60]}..."
        )
        rows.append(
            {
                "query": item["query"],
                "category": item["category"],
                "retrieval_hit": retrieval_hit,
                "top_distance": round(top_distance, 4),
                "retrieved_ids": retrieved_ids,
                "answer": answer[:120],
                "faithfulness": scores["faithfulness"],
                "correctness": scores["correctness"],
                "reason": scores["reason"],
            }
        )
    return pd.DataFrame(rows)


print("=== Running RAG Pipeline Evaluation ===\n")
rag_eval = evaluate_rag(book_collection, rag_golden)

=== Running RAG Pipeline Evaluation ===

  [HIT] What is the new AI policy?                         | Faith=5 Corr=5 | The new AI policy, effective June 1st, requires all employee...
  [HIT] What were the Q2 financial results?                | Faith=5 Corr=5 | Q2 financial results showed a 15% increase in revenue, drive...
  [HIT] When is the Aurora Project beta release?           | Faith=5 Corr=5 | The Aurora Project beta release is scheduled for the third q...
  [HIT] How do I submit expense reports?                   | Faith=5 Corr=5 | Submit expense reports through the new online portal by the ...
  [HIT] What is our remote work policy?                    | Faith=5 Corr=5 | I cannot find this information....


In [9]:
# === RAG Step 4: Summary ===
# Roll up per-question scores into three headline numbers.

print("=" * 60)
print("HAND-BUILT RAG EVALUATION SUMMARY")
print("=" * 60)
ret_acc = rag_eval["retrieval_hit"].mean() * 100
avg_faith = rag_eval["faithfulness"].mean()
avg_corr = rag_eval["correctness"].mean()
print(f"  Retrieval accuracy:  {ret_acc:.0f}% ({rag_eval['retrieval_hit'].sum()}/{len(rag_eval)})")
print(f"  Avg faithfulness:    {avg_faith:.2f} / 5")
print(f"  Avg correctness:     {avg_corr:.2f} / 5")

misses = rag_eval[~rag_eval["retrieval_hit"]]
if len(misses) > 0:
    print(f"\n  Retrieval misses ({len(misses)}):")
    for _, r in misses.iterrows():
        print(f"    Q: {r['query']}  |  Got: {r['retrieved_ids']}")

low = rag_eval[rag_eval["faithfulness"] < 4]
if len(low) > 0:
    print(f"\n  Low faithfulness ({len(low)}):")
    for _, r in low.iterrows():
        print(f"    Q: {r['query']}  |  {r['reason']}")

print()
print("When retrieval is correct but correctness drops, fix the prompt.")
print("When retrieval misses, fix the data strategy.")

HAND-BUILT RAG EVALUATION SUMMARY
  Retrieval accuracy:  100% (5/5)
  Avg faithfulness:    5.00 / 5
  Avg correctness:     5.00 / 5

When retrieval is correct but correctness drops, fix the prompt.
When retrieval misses, fix the data strategy.


In [10]:
# === RAG Step 5: RAGAS Automated Metrics ===
# RAGAS provides four standardized metrics on a 0-1 scale:
#   faithfulness, answer_relevancy, context_precision, context_recall

# Patch: RAGAS expects langchain_community.chat_models.vertexai which was
# removed in newer langchain-community. Create a shim so the import succeeds.
import types, importlib
try:
    importlib.import_module("langchain_community.chat_models.vertexai")
except (ImportError, ModuleNotFoundError):
    try:
        from langchain_google_vertexai import ChatVertexAI
        import langchain_community.chat_models as _cm
        shim = types.ModuleType("langchain_community.chat_models.vertexai")
        shim.ChatVertexAI = ChatVertexAI
        sys.modules["langchain_community.chat_models.vertexai"] = shim
        print("Patched vertexai shim for RAGAS compatibility")
    except ImportError:
        pass  # will be caught below

RAGAS_AVAILABLE = False
try:
    from ragas import evaluate as ragas_evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    from ragas import EvaluationDataset, SingleTurnSample
    RAGAS_AVAILABLE = True
    print("RAGAS loaded successfully")
except ImportError:
    try:
        from ragas import evaluate as ragas_evaluate
        from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
        from datasets import Dataset
        RAGAS_AVAILABLE = True
        print("RAGAS loaded (legacy API)")
    except ImportError as e:
        print(f"RAGAS not available: {e}")
        print("Install manually with: pip install ragas langchain-openai langchain-community langchain-google-vertexai")
        print("Skipping RAGAS evaluation -- the hand-built metrics above still apply.")

if RAGAS_AVAILABLE:
    samples = []
    for item in rag_golden:
        if item["expected_doc_id"] is None:
            continue

        ret = book_collection.query(
            query_texts=[item["query"]], n_results=2, include=["documents"]
        )
        contexts = ret["documents"][0]

        ctx = "\n---\n".join(contexts)
        prompt = (
            "Use ONLY the following context to answer. "
            "If the answer is not in the context, say so.\n\n"
            f"<context>{ctx}</context>\n"
            f"<question>{item['query']}</question>\nAnswer:"
        )
        resp = client.chat.completions.create(
            model=BASE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        answer = resp.choices[0].message.content.strip()

        try:
            sample = SingleTurnSample(
                user_input=item["query"],
                response=answer,
                retrieved_contexts=contexts,
                reference=item["reference_answer"],
            )
            samples.append(sample)
        except Exception:
            samples.append({
                "question": item["query"],
                "answer": answer,
                "contexts": contexts,
                "ground_truth": item["reference_answer"],
            })

    print(f"Prepared {len(samples)} samples for RAGAS\n")

    # Run only metrics that do not require embeddings (answer_relevancy needs
    # OpenAIEmbeddings.embed_query which breaks on newer langchain-openai).
    # Skip answer_relevancy to avoid the embed_query AttributeError.
    safe_metrics = [faithfulness, context_precision, context_recall]
    try:
        dataset = EvaluationDataset(samples=samples)
        ragas_result = ragas_evaluate(
            dataset=dataset,
            metrics=safe_metrics,
        )
    except Exception:
        try:
            from datasets import Dataset
            ds = Dataset.from_list(samples)
            ragas_result = ragas_evaluate(
                ds, metrics=safe_metrics
            )
        except Exception as e2:
            print(f"RAGAS evaluation failed: {e2}")
            ragas_result = None

    if ragas_result is not None:
        print("=== RAGAS Evaluation Results ===")
        # EvaluationResult uses to_pandas(), not .items()
        try:
            df = ragas_result.to_pandas()
            for col in df.columns:
                if col in ("user_input", "response", "retrieved_contexts", "reference"):
                    continue
                vals = pd.to_numeric(df[col], errors="coerce").dropna()
                if len(vals) > 0:
                    score = vals.mean()
                    bar = "#" * int(score * 20)
                    print(f"  {col:25s} {score:.3f}  {bar}")
        except Exception:
            # Fallback: try dict-like access
            for m in safe_metrics:
                name = m.name if hasattr(m, "name") else str(m)
                try:
                    score = ragas_result[name]
                    if isinstance(score, (int, float)):
                        bar = "#" * int(score * 20)
                        print(f"  {name:25s} {score:.3f}  {bar}")
                except Exception:
                    pass

        print("\nMetric guide:")
        print("  faithfulness       -- 1.0 = answer uses only retrieved context")
        print("  context_precision  -- 1.0 = every retrieved chunk is relevant")
        print("  context_recall     -- 1.0 = all needed information was retrieved")
        print("\n  (answer_relevancy skipped -- requires embedding model compatibility)")

Patched vertexai shim for RAGAS compatibility
RAGAS loaded successfully
Prepared 4 samples for RAGAS



Evaluating: 100%|██████████| 12/12 [00:27<00:00,  2.30s/it]

=== RAGAS Evaluation Results ===
  faithfulness              1.000  ####################
  context_precision         1.000  ###################
  context_recall            1.000  ####################

Metric guide:
  faithfulness       -- 1.0 = answer uses only retrieved context
  context_precision  -- 1.0 = every retrieved chunk is relevant
  context_recall     -- 1.0 = all needed information was retrieved

  (answer_relevancy skipped -- requires embedding model compatibility)


In [11]:
# === RAG Step 6: Combined Report ===
# Compare hand-built and RAGAS results side by side.

print("=" * 60)
print("RAG PIPELINE EVALUATION -- COMBINED REPORT")
print("=" * 60)

print("\n--- Hand-Built Metrics (our code) ---")
print(f"  Retrieval accuracy:  {ret_acc:.0f}%")
print(f"  Avg faithfulness:    {avg_faith:.2f} / 5  (LLM-judge, 5-point scale)")
print(f"  Avg correctness:     {avg_corr:.2f} / 5  (LLM-judge, 5-point scale)")

if RAGAS_AVAILABLE and ragas_result is not None:
    print("\n--- RAGAS Metrics (automated framework) ---")
    try:
        df = ragas_result.to_pandas()
        for col in df.columns:
            if col in ("user_input", "response", "retrieved_contexts", "reference"):
                continue
            vals = pd.to_numeric(df[col], errors="coerce").dropna()
            if len(vals) > 0:
                print(f"  {col:25s} {vals.mean():.3f}  (0-1 scale)")
    except Exception:
        for m in [faithfulness, context_precision, context_recall]:
            name = m.name if hasattr(m, "name") else str(m)
            try:
                score = ragas_result[name]
                if isinstance(score, (int, float)):
                    print(f"  {name:25s} {score:.3f}  (0-1 scale)")
            except Exception:
                pass

print("\n--- Per-Question Breakdown ---")
for _, row in rag_eval.iterrows():
    status = "PASS" if row["faithfulness"] >= 4 and row["correctness"] >= 4 else "FLAG"
    print(f"  [{status}] {row['query'][:45]:45s} Faith={row['faithfulness']} Corr={row['correctness']}")

print()
print("Now that we know the RAG pipeline works, we can evaluate the data")
print("improvements from Chapter 6 (synthetic data) and Chapter 7 (fine-tuning).")

RAG PIPELINE EVALUATION -- COMBINED REPORT

--- Hand-Built Metrics (our code) ---
  Retrieval accuracy:  100%
  Avg faithfulness:    5.00 / 5  (LLM-judge, 5-point scale)
  Avg correctness:     5.00 / 5  (LLM-judge, 5-point scale)

--- RAGAS Metrics (automated framework) ---
  faithfulness              1.000  (0-1 scale)
  context_precision         1.000  (0-1 scale)
  context_recall            1.000  (0-1 scale)

--- Per-Question Breakdown ---
  [PASS] What is the new AI policy?                    Faith=5 Corr=5
  [PASS] What were the Q2 financial results?           Faith=5 Corr=5
  [PASS] When is the Aurora Project beta release?      Faith=5 Corr=5
  [PASS] How do I submit expense reports?              Faith=5 Corr=5
  [PASS] What is our remote work policy?               Faith=5 Corr=5

Now that we know the RAG pipeline works, we can evaluate the data
improvements from Chapter 6 (synthetic data) and Chapter 7 (fine-tuning).


## Part 1b: Beyond Accuracy -- Multi-Metric Evaluation

Part 0 scored each response on a single number: how well it matched the reference answer. That catches obvious failures but misses subtler problems. A response can be factually correct and still confuse the reader, use broken logic, or sound unprofessional.

At Clearwater Analytics, Dan Siddall built an evaluation rubric with eight attributes. We apply four of them here (the four the basic judge misses) to the same Prompt B responses from Part 0. The question: did the basic judge miss anything important?

**The four new metrics:**
- **Coherence** -- is the answer logically structured, or are facts dumped in a confusing order?
- **Reasoning** -- did the model reach the answer through sound logic, or guess right by accident?
- **Professionalism** -- is the tone appropriate for the domain?
- **Ethical Considerations** -- any bias, privacy leaks, or harmful content?

In [12]:
# === Step 1: Define the Multi-Metric Judge ===
# The basic judge from Part 0 scored on accuracy alone (1-5).
# This judge scores on all eight attributes from Siddall's framework.

def multi_metric_judge(query, response, reference_answer):
    """Score a response on 8 attributes. Returns dict of scores + reasons."""

    prompt = f"""You are an evaluation judge. Score this response on eight attributes (1-5 each).
You MUST provide a one-sentence justification for each score.

Question: {query}
Reference answer: {reference_answer}
Model response: {response}

Attributes:
1. Accuracy: Does the response match the reference answer?
2. Relevance: Does the response directly address the question asked?
3. Coherence: Is the response logically structured and internally consistent?
4. Context adherence: Does the response stick to provided information without adding fabricated details?
5. Reasoning: Does the model support its statements with sound logic?
6. Professionalism: Is the tone appropriate for a customer support context?
7. Ethical considerations: Is the response free from bias, privacy issues, or harmful content?
8. Creativity: Does the response go beyond template answers when appropriate?

Return ONLY valid JSON:
{{"accuracy": N, "accuracy_reason": "...",
  "relevance": N, "relevance_reason": "...",
  "coherence": N, "coherence_reason": "...",
  "context_adherence": N, "context_adherence_reason": "...",
  "reasoning": N, "reasoning_reason": "...",
  "professionalism": N, "professionalism_reason": "...",
  "ethical": N, "ethical_reason": "...",
  "creativity": N, "creativity_reason": "..."}}"""

    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    content = resp.choices[0].message.content.strip()
    if '```json' in content:
        content = content.split('```json')[1].split('```')[0].strip()
    elif '```' in content:
        content = content.split('```')[1].split('```')[0].strip()
    return json.loads(content)

print("Multi-metric judge defined.")
print("Scores on 8 attributes: accuracy, relevance, coherence, context adherence,")
print("reasoning, professionalism, ethical considerations, creativity.")

Multi-metric judge defined.
Scores on 8 attributes: accuracy, relevance, coherence, context adherence,
reasoning, professionalism, ethical considerations, creativity.


In [13]:
# === Step 2: Re-run Prompt B responses and score with all 8 metrics ===
# Prompt B (no context) gave generic answers in Part 0.
# The basic judge scored them 2-4 out of 5 on accuracy alone.
# Let's see what the full rubric reveals.

print("=== Scoring Prompt B responses on all 8 attributes ===\n")

multi_results_b = []
for item in golden_dataset:
    # Generate Prompt B response (no context)
    prompt = PROMPT_B.format(query=item['query'])
    resp = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    model_answer = resp.choices[0].message.content.strip()

    # Score with multi-metric judge
    scores = multi_metric_judge(item['query'], model_answer, item['reference_answer'])
    scores['query'] = item['query'][:50]
    scores['answer_preview'] = model_answer[:100]
    multi_results_b.append(scores)

    print(f"Q: {item['query'][:60]}")
    print(f"  Accuracy={scores['accuracy']}  Relevance={scores['relevance']}  "
          f"Coherence={scores['coherence']}  Reasoning={scores['reasoning']}")
    print(f"  Context adherence={scores['context_adherence']}  "
          f"Professionalism={scores['professionalism']}  "
          f"Ethical={scores['ethical']}  Creativity={scores['creativity']}")
    print()

multi_b_df = pd.DataFrame(multi_results_b)
print("Done. Results stored in multi_b_df.")

=== Scoring Prompt B responses on all 8 attributes ===

Q: What is the refund policy for digital products?
  Accuracy=2  Relevance=4  Coherence=5  Reasoning=3
  Context adherence=2  Professionalism=5  Ethical=5  Creativity=2

Q: Can I return a physical item after 30 days?
  Accuracy=4  Relevance=5  Coherence=5  Reasoning=4
  Context adherence=3  Professionalism=5  Ethical=5  Creativity=3

Q: What happens if my subscription renews and I want to cancel?
  Accuracy=3  Relevance=5  Coherence=5  Reasoning=4
  Context adherence=3  Professionalism=5  Ethical=5  Creativity=3

Q: I bought a gift card and the recipient lost it. Can I get a 
  Accuracy=3  Relevance=5  Coherence=5  Reasoning=4
  Context adherence=3  Professionalism=5  Ethical=5  Creativity=3

Q: My order arrived damaged. Who pays for return shipping?
  Accuracy=5  Relevance=5  Coherence=5  Reasoning=4
  Context adherence=3  Professionalism=5  Ethical=5  Creativity=4

Done. Results stored in multi_b_df.


In [14]:
# === Step 3: Now score Prompt A (with context) on the same 8 metrics ===
# Prompt A had context and scored well on accuracy in Part 0.
# Does it also score well on coherence, reasoning, professionalism?

print("=== Scoring Prompt A responses on all 8 attributes ===\n")

multi_results_a = []
for item in golden_dataset:
    # Generate Prompt A response (with context)
    prompt = PROMPT_A.format(query=item['query'], reference=item['reference_answer'])
    resp = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    model_answer = resp.choices[0].message.content.strip()

    # Score with multi-metric judge
    scores = multi_metric_judge(item['query'], model_answer, item['reference_answer'])
    scores['query'] = item['query'][:50]
    scores['answer_preview'] = model_answer[:100]
    multi_results_a.append(scores)

    print(f"Q: {item['query'][:60]}")
    print(f"  Accuracy={scores['accuracy']}  Relevance={scores['relevance']}  "
          f"Coherence={scores['coherence']}  Reasoning={scores['reasoning']}")
    print(f"  Context adherence={scores['context_adherence']}  "
          f"Professionalism={scores['professionalism']}  "
          f"Ethical={scores['ethical']}  Creativity={scores['creativity']}")
    print()

multi_a_df = pd.DataFrame(multi_results_a)
print("Done. Results stored in multi_a_df.")

=== Scoring Prompt A responses on all 8 attributes ===

Q: What is the refund policy for digital products?
  Accuracy=5  Relevance=5  Coherence=5  Reasoning=5
  Context adherence=5  Professionalism=5  Ethical=5  Creativity=3

Q: Can I return a physical item after 30 days?
  Accuracy=5  Relevance=5  Coherence=5  Reasoning=5
  Context adherence=5  Professionalism=5  Ethical=5  Creativity=3

Q: What happens if my subscription renews and I want to cancel?
  Accuracy=5  Relevance=5  Coherence=5  Reasoning=5
  Context adherence=5  Professionalism=5  Ethical=5  Creativity=3

Q: I bought a gift card and the recipient lost it. Can I get a 
  Accuracy=5  Relevance=5  Coherence=5  Reasoning=5
  Context adherence=5  Professionalism=5  Ethical=5  Creativity=3

Q: My order arrived damaged. Who pays for return shipping?
  Accuracy=5  Relevance=5  Coherence=5  Reasoning=5
  Context adherence=5  Professionalism=5  Ethical=5  Creativity=3

Done. Results stored in multi_a_df.


In [15]:
# === Step 4: Compare Basic Score vs Multi-Metric Score ===
# The key question: did the extra metrics catch problems that accuracy alone missed?

metrics = ['accuracy', 'relevance', 'coherence', 'context_adherence',
           'reasoning', 'professionalism', 'ethical', 'creativity']

print('=' * 70)
print('MULTI-METRIC COMPARISON: Prompt A (with context) vs Prompt B (no context)')
print('=' * 70)
print()
print(f'{"Metric":<22s} {"Prompt A":>10s} {"Prompt B":>10s} {"Delta":>10s}  {"Winner":>8s}')
print('-' * 70)

for m in metrics:
    avg_a = multi_a_df[m].mean()
    avg_b = multi_b_df[m].mean()
    delta = avg_a - avg_b
    winner = 'A' if delta > 0.1 else ('B' if delta < -0.1 else 'tie')
    flag = ' <<<' if abs(delta) > 1.0 else ''
    print(f'{m:<22s} {avg_a:>10.2f} {avg_b:>10.2f} {delta:>+10.2f}  {winner:>8s}{flag}')

print()
print('Key insight: look for metrics where Prompt B scores surprisingly well on')
print('accuracy but poorly on context_adherence or coherence. That is the gap')
print('the basic judge would have missed.')
print()

# Find the biggest gaps per question
print('Per-question biggest gap (Prompt B):')
for i, row in multi_b_df.iterrows():
    scores = {m: row[m] for m in metrics}
    worst_metric = min(scores, key=scores.get)
    best_metric = max(scores, key=scores.get)
    if scores[best_metric] - scores[worst_metric] >= 2:
        print(f'  {row["query"]}')
        print(f'    Best:  {best_metric}={scores[best_metric]}  |  Worst: {worst_metric}={scores[worst_metric]}')
        print(f'    Reason: {row.get(worst_metric + "_reason", "n/a")}')

MULTI-METRIC COMPARISON: Prompt A (with context) vs Prompt B (no context)

Metric                   Prompt A   Prompt B      Delta    Winner
----------------------------------------------------------------------
accuracy                     5.00       3.40      +1.60         A <<<
relevance                    5.00       4.80      +0.20         A
coherence                    5.00       5.00      +0.00       tie
context_adherence            5.00       2.80      +2.20         A <<<
reasoning                    5.00       3.80      +1.20         A <<<
professionalism              5.00       5.00      +0.00       tie
ethical                      5.00       5.00      +0.00       tie
creativity                   3.00       3.00      +0.00       tie

Key insight: look for metrics where Prompt B scores surprisingly well on
accuracy but poorly on context_adherence or coherence. That is the gap
the basic judge would have missed.

Per-question biggest gap (Prompt B):
  What is the refund policy fo

In [16]:
# === Step 5: Summary -- What the Extra Metrics Taught Us ===

print('=' * 60)
print('WHAT THE EXTRA METRICS CAUGHT')
print('=' * 60)
print()

# Context adherence gap: Prompt B fabricates details the basic judge missed
ctx_gap = multi_a_df['context_adherence'].mean() - multi_b_df['context_adherence'].mean()
print(f'1. Context adherence gap: {ctx_gap:+.2f}')
print(f'   Prompt A: {multi_a_df["context_adherence"].mean():.2f}  Prompt B: {multi_b_df["context_adherence"].mean():.2f}')
if ctx_gap > 0.5:
    print('   Prompt B fabricated details. The basic accuracy judge gave partial')
    print('   credit because the fabricated details sounded plausible.')
print()

# Coherence: do the no-context answers ramble?
coh_gap = multi_a_df['coherence'].mean() - multi_b_df['coherence'].mean()
print(f'2. Coherence gap: {coh_gap:+.2f}')
print(f'   Prompt A: {multi_a_df["coherence"].mean():.2f}  Prompt B: {multi_b_df["coherence"].mean():.2f}')
if coh_gap > 0.5:
    print('   Without context, the model padded its response with generic advice.')
    print('   Accurate enough to score 3/5, but incoherent as customer support.')
print()

# Professionalism: does tone differ?
prof_gap = multi_a_df['professionalism'].mean() - multi_b_df['professionalism'].mean()
print(f'3. Professionalism gap: {prof_gap:+.2f}')
print(f'   Prompt A: {multi_a_df["professionalism"].mean():.2f}  Prompt B: {multi_b_df["professionalism"].mean():.2f}')
print()

# Overall takeaway
print('TAKEAWAY:')
print('A single accuracy score hides these problems. If you had only looked')
print('at the basic judge from Part 0, Prompt B would look "mostly okay" at')
print(f'{results_b["score"].mean():.1f}/5. The multi-metric judge shows the full picture:')
print(f'context adherence at {multi_b_df["context_adherence"].mean():.1f}/5 and ')
print(f'coherence at {multi_b_df["coherence"].mean():.1f}/5 tell a different story.')
print()
print('This is why evaluation rubrics need more than one dimension.')

WHAT THE EXTRA METRICS CAUGHT

1. Context adherence gap: +2.20
   Prompt A: 5.00  Prompt B: 2.80
   Prompt B fabricated details. The basic accuracy judge gave partial
   credit because the fabricated details sounded plausible.

2. Coherence gap: +0.00
   Prompt A: 5.00  Prompt B: 5.00

3. Professionalism gap: +0.00
   Prompt A: 5.00  Prompt B: 5.00

TAKEAWAY:
A single accuracy score hides these problems. If you had only looked
at the basic judge from Part 0, Prompt B would look "mostly okay" at
3.6/5. The multi-metric judge shows the full picture:
context adherence at 2.8/5 and 
coherence at 5.0/5 tell a different story.

This is why evaluation rubrics need more than one dimension.


## Part 2: Loading Chapter 6 Datasets

Chapter 6 produced four synthetic datasets: HR QA pairs, structured QA pairs, preference pairs, and augmented text. We load all four and inspect them before evaluation.

In [ ]:
# Load all four Chapter 6 datasets
# Build the path from repo_root (set in Setup) so it works both locally and on
# Google Colab. A hardcoded '../../chapter_06' only works when the current
# directory happens to be chapter_09/Jupyter_Notebooks/; on Colab it is not.
ch6_path = repo_root / 'chapter_06' / 'datasets'
if not ch6_path.exists():
    raise FileNotFoundError(
        f"Chapter 6 datasets not found at {ch6_path}.\n"
        "Run the Chapter 6 notebook first (it writes these CSVs), or make sure the "
        "repository is cloned and the Setup cell found the correct repo_root."
    )

hr_qa = pd.read_csv(ch6_path / 'hr_policy_qa_dataset.csv')
structured_qa = pd.read_csv(ch6_path / 'structured_qa_dataset.csv')
preference_pairs = pd.read_csv(ch6_path / 'preference_pairs_dataset.csv')
augmented_text = pd.read_csv(ch6_path / 'augmented_text_dataset.csv')

print('Chapter 6 datasets loaded:')
print(f'  HR QA pairs:        {len(hr_qa)} rows')
print(f'  Structured QA:      {len(structured_qa)} rows')
print(f'  Preference pairs:   {len(preference_pairs)} rows')
print(f'  Augmented text:     {len(augmented_text)} rows')
print()
print('HR QA sample:')
hr_qa.head(3)

Chapter 6 datasets loaded:
  HR QA pairs:        6 rows
  Structured QA:      12 rows
  Preference pairs:   4 rows
  Augmented text:     25 rows

HR QA sample:


,question,answer,source_policy
0,How many days of paid vacation do employees re...,Employees receive 15 days of paid vacation per...,POL-001 - Vacation Policy
1,How far in advance should employees request va...,Employees should request vacation time at leas...,POL-001 - Vacation Policy
2,How many days per week can employees work remo...,Employees can work remotely up to 3 days per w...,POL-002 - Remote Work Policy


## Part 3: Evaluating Synthetic QA Quality

The first question about any synthetic dataset: is it faithful to the source?
A generated answer that sounds plausible but adds details not in the source policy is a hallucination.
We use LLM-as-a-judge to score each QA pair on three dimensions.

In [18]:
def judge_qa_faithfulness(question, answer, source):
    """Score a QA pair on faithfulness, relevance, and accuracy using LLM-as-judge.
    Returns dict with scores 1-5 and justification.
    This implements the Siddall (2024) evaluation framework from the chapter."""

    prompt = f"""You are an evaluation judge. Score this QA pair on three dimensions.
Each score is 1-5 where 5 is best.

Source policy: {source}
Question: {question}
Answer: {answer}

Score these three dimensions:
1. Faithfulness: Does the answer ONLY use information from the source? (5 = fully grounded, 1 = hallucinated)
2. Relevance: Does the answer address the question? (5 = directly answers, 1 = off topic)
3. Accuracy: Is the answer factually correct given the source? (5 = fully correct, 1 = wrong)

Return ONLY valid JSON:
{{"faithfulness": N, "relevance": N, "accuracy": N, "justification": "one sentence"}}"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )

    content = response.choices[0].message.content.strip()
    # Extract JSON
    if '```json' in content:
        content = content.split('```json')[1].split('```')[0].strip()
    elif '```' in content:
        content = content.split('```')[1].split('```')[0].strip()
    return json.loads(content)


# Evaluate all HR QA pairs
print('Evaluating HR QA pairs with LLM-as-judge...')
print()

results = []
for i, row in hr_qa.iterrows():
    scores = judge_qa_faithfulness(
        row['question'],
        row['answer'],
        row['source_policy']
    )
    scores['question'] = row['question'][:60]
    results.append(scores)
    print(f"  [{i+1}/{len(hr_qa)}] Faith={scores['faithfulness']} Rel={scores['relevance']} Acc={scores['accuracy']}")

eval_df = pd.DataFrame(results)

print()
print('=== Faithfulness Report ===')
print(f"Average faithfulness: {eval_df['faithfulness'].mean():.2f} / 5.00")
print(f"Average relevance:   {eval_df['relevance'].mean():.2f} / 5.00")
print(f"Average accuracy:    {eval_df['accuracy'].mean():.2f} / 5.00")
print(f"Pairs scoring below 4 on faithfulness: {(eval_df['faithfulness'] < 4).sum()}")
print()

# Show any problematic pairs
low_faith = eval_df[eval_df['faithfulness'] < 4]
if len(low_faith) > 0:
    print('Potentially hallucinated answers:')
    for _, row in low_faith.iterrows():
        print(f"  Q: {row['question']}")
        print(f"  Justification: {row['justification']}")
        print()
else:
    print('All QA pairs are faithful to the source. Good.')

Evaluating HR QA pairs with LLM-as-judge...

  [1/6] Faith=1 Rel=5 Acc=1
  [2/6] Faith=5 Rel=5 Acc=5
  [3/6] Faith=5 Rel=5 Acc=5
  [4/6] Faith=5 Rel=5 Acc=5
  [5/6] Faith=5 Rel=5 Acc=5
  [6/6] Faith=1 Rel=5 Acc=1

=== Faithfulness Report ===
Average faithfulness: 3.67 / 5.00
Average relevance:   5.00 / 5.00
Average accuracy:    3.67 / 5.00
Pairs scoring below 4 on faithfulness: 2

Potentially hallucinated answers:
  Q: How many days of paid vacation do employees receive per year
  Justification: The answer directly addresses the question but the provided source contains no information confirming that employees receive 15 vacation days.

  Q: When is a doctor note required for sick leave?
  Justification: The answer directly addresses the question, but the provided source text does not include the 3-day requirement, so the claim is unsupported and cannot be verified as accurate.



## Part 4: Evaluating QA Diversity

Faithful answers are useless if every question is a paraphrase of the same thing.
We measure diversity with pure Python, no API calls needed.
This is a rule-based metric, exactly the kind ROUGE and BLEU belong to.

In [19]:
def measure_diversity(questions):
    """Measure lexical diversity of a set of questions.
    Higher unique n-gram ratio = more diverse.
    Self-BLEU measures how similar questions are to each other (lower = better)."""

    all_unigrams = []
    all_bigrams = []

    for q in questions:
        tokens = q.lower().split()
        all_unigrams.extend(tokens)
        all_bigrams.extend(zip(tokens[:-1], tokens[1:]))

    unique_unigrams = len(set(all_unigrams))
    unique_bigrams = len(set(all_bigrams))
    total_unigrams = len(all_unigrams)
    total_bigrams = len(all_bigrams)

    # Type-token ratio
    ttr = unique_unigrams / total_unigrams if total_unigrams > 0 else 0
    bigram_ttr = unique_bigrams / total_bigrams if total_bigrams > 0 else 0

    # Question-start diversity: how many unique first 3 words?
    starters = [' '.join(q.lower().split()[:3]) for q in questions]
    starter_diversity = len(set(starters)) / len(starters) if starters else 0

    return {
        'total_questions': len(questions),
        'unique_unigrams': unique_unigrams,
        'unigram_ttr': round(ttr, 3),
        'unique_bigrams': unique_bigrams,
        'bigram_ttr': round(bigram_ttr, 3),
        'starter_diversity': round(starter_diversity, 3),
        'unique_starters': len(set(starters))
    }


# Measure diversity for both QA datasets
print('=== HR QA Diversity ===')
hr_div = measure_diversity(hr_qa['question'].tolist())
for k, v in hr_div.items():
    print(f'  {k}: {v}')

print()
print('=== Structured QA Diversity ===')
str_div = measure_diversity(structured_qa['question'].tolist())
for k, v in str_div.items():
    print(f'  {k}: {v}')

print()
# Interpretation
if hr_div['starter_diversity'] < 0.5:
    print('WARNING: Over half the HR questions start the same way.')
    print('The model may learn to pattern-match question openers instead of understanding the question.')
    print('Fix: regenerate with explicit diversity instructions in the prompt.')
else:
    print('Question starters are reasonably diverse. Good.')

if hr_div['unigram_ttr'] < 0.3:
    print('WARNING: Low vocabulary diversity. Questions reuse the same words heavily.')
else:
    print('Vocabulary diversity is acceptable.')

=== HR QA Diversity ===
  total_questions: 6
  unique_unigrams: 50
  unigram_ttr: 0.746
  unique_bigrams: 55
  bigram_ttr: 0.902
  starter_diversity: 0.667
  unique_starters: 4

=== Structured QA Diversity ===
  total_questions: 12
  unique_unigrams: 28
  unigram_ttr: 0.333
  unique_bigrams: 36
  bigram_ttr: 0.5
  starter_diversity: 0.417
  unique_starters: 5

Question starters are reasonably diverse. Good.
Vocabulary diversity is acceptable.


## Part 5: Evaluating Preference Pairs with Position Bias Check

Chapter 6 generated preferred vs non-preferred response pairs.
Now we test: can an LLM judge correctly identify which is better?
And does the judge show position bias, the tendency to favor whichever response comes first?
Siddall (2024) flagged this exact problem.

In [20]:
import random

def judge_preference(instruction, response_a, response_b, a_is_preferred):
    """Ask LLM judge to pick the better response.
    We randomize order to detect position bias.
    Returns dict with judge choice and whether it matches the label."""

    # Randomize presentation order to test for position bias
    show_preferred_first = random.random() < 0.5

    if show_preferred_first:
        first, second = response_a, response_b
        correct_choice = 'A'
    else:
        first, second = response_b, response_a
        correct_choice = 'B'

    prompt = f"""Which response is better for this instruction? Answer with just A or B.

Instruction: {instruction}

Response A: {first}

Response B: {second}

Better response (A or B):"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=5
    )

    judge_pick = response.choices[0].message.content.strip().upper()
    # Extract just A or B
    if 'A' in judge_pick and 'B' not in judge_pick:
        judge_pick = 'A'
    elif 'B' in judge_pick and 'A' not in judge_pick:
        judge_pick = 'B'
    else:
        judge_pick = judge_pick[0] if judge_pick else '?'

    return {
        'correct': judge_pick == correct_choice,
        'judge_pick': judge_pick,
        'preferred_shown_first': show_preferred_first,
        'picked_first': judge_pick == 'A'
    }


print('Evaluating preference pairs with position bias check...')
print()

pref_results = []
for i, row in preference_pairs.iterrows():
    result = judge_preference(
        row['instruction'],
        row['preferred_response'],
        row['non_preferred_response'],
        a_is_preferred=True
    )
    pref_results.append(result)
    status = 'correct' if result['correct'] else 'WRONG'
    print(f"  [{i+1}/{len(preference_pairs)}] Judge picked: {result['judge_pick']} ({status})")

pref_df = pd.DataFrame(pref_results)

accuracy = pref_df['correct'].mean()
position_bias = pref_df['picked_first'].mean()

print()
print('=== Preference Evaluation Report ===')
print(f'Judge accuracy: {accuracy:.1%} ({pref_df["correct"].sum()}/{len(pref_df)} correct)')
print(f'Position bias:  {position_bias:.1%} picked Response A (first position)')
print(f'  (50% = no bias, >70% = strong position bias)')
print()

if accuracy < 0.7:
    print('WARNING: Judge accuracy below 70%. Either the preference pairs are too subtle')
    print('or the judge model is not strong enough. Try a stronger judge or clearer pairs.')
if position_bias > 0.7:
    print('WARNING: Strong position bias detected. The judge favors whichever response')
    print('appears first. This is exactly what Siddall (2024) warned about.')
    print('Fix: always randomize order and average across orderings.')

Evaluating preference pairs with position bias check...

  [1/4] Judge picked: B (correct)
  [2/4] Judge picked: B (correct)
  [3/4] Judge picked: B (correct)
  [4/4] Judge picked: A (correct)

=== Preference Evaluation Report ===
Judge accuracy: 100.0% (4/4 correct)
Position bias:  25.0% picked Response A (first position)
  (50% = no bias, >70% = strong position bias)



## Part 6: Loading and Inspecting Chapter 7 SFT Data

Chapter 7 fine-tuned a model using SFT data. Before we evaluate the model,
we check the training data itself. Bad training data produces bad models
regardless of how well the fine-tuning algorithm works.

In [ ]:
# Load Chapter 7 SFT datasets
# repo_root-relative (set in Setup) so it works locally and on Google Colab.
ch7_path = repo_root / 'chapter_07' / 'datasets'
if not ch7_path.exists():
    raise FileNotFoundError(
        f"Chapter 7 datasets not found at {ch7_path}.\n"
        "Run the Chapter 7 notebook first (it writes sft_train.jsonl / sft_valid.jsonl), "
        "or make sure the repository is cloned and the Setup cell found the correct repo_root."
    )

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return records

sft_train = load_jsonl(ch7_path / 'sft_train.jsonl')
sft_valid = load_jsonl(ch7_path / 'sft_valid.jsonl')

print(f'SFT training examples:   {len(sft_train)}')
print(f'SFT validation examples: {len(sft_valid)}')
print()

# Check behavioral consistency: do all examples use the same system prompt?
system_prompts = set()
for record in sft_train + sft_valid:
    for msg in record['messages']:
        if msg['role'] == 'system':
            system_prompts.add(msg['content'])

print(f'Unique system prompts: {len(system_prompts)}')
if len(system_prompts) == 1:
    print('Good: all examples share one behavioral contract.')
    print(f'System prompt: "{list(system_prompts)[0][:100]}..."')
else:
    print('WARNING: Multiple system prompts found. This can cause conflicting behavior.')
    for sp in system_prompts:
        print(f'  - "{sp[:80]}..."')

print()

# Check output structure consistency
print('=== Output Structure Check ===')
has_summary = 0
has_next_steps = 0
has_risks = 0

for record in sft_train:
    assistant_msg = [m for m in record['messages'] if m['role'] == 'assistant'][0]['content']
    if 'Summary' in assistant_msg or 'summary' in assistant_msg:
        has_summary += 1
    if 'Next steps' in assistant_msg or 'next steps' in assistant_msg:
        has_next_steps += 1
    if 'Risks' in assistant_msg or 'risks' in assistant_msg:
        has_risks += 1

print(f'Examples with Summary section:    {has_summary}/{len(sft_train)}')
print(f'Examples with Next Steps section: {has_next_steps}/{len(sft_train)}')
print(f'Examples with Risks section:      {has_risks}/{len(sft_train)}')
print()

consistency = min(has_summary, has_next_steps, has_risks) / len(sft_train)
if consistency > 0.8:
    print(f'Output structure consistency: {consistency:.0%}. The behavioral contract is clear.')
else:
    print(f'Output structure consistency: {consistency:.0%}. Some examples deviate from the contract.')
    print('This is exactly the kind of inconsistency that Chapter 7 warns about.')

SFT training examples:   10
SFT validation examples: 3

Unique system prompts: 1
Good: all examples share one behavioral contract.
System prompt: "You are a policy-aware support assistant. If context is insufficient, say 'Information not available..."

=== Output Structure Check ===
Examples with Summary section:    10/10
Examples with Next Steps section: 10/10
Examples with Risks section:      10/10

Output structure consistency: 100%. The behavioral contract is clear.


## Part 7: Evaluating Base Model vs the Behavioral Contract

We take the validation prompts from Chapter 7 and run them through the base model.
Then we score the outputs against the behavioral contract.
This is the baseline before fine-tuning.

In [22]:
def evaluate_against_contract(messages, model_name):
    """Run a prompt through a model and score the output against the behavioral contract.
    The contract from Chapter 7 SFT data: Summary, Next Steps, Risks sections."""

    # Get model response
    system_msg = [m for m in messages if m['role'] == 'system']
    user_msg = [m for m in messages if m['role'] == 'user']

    response = client.chat.completions.create(
        model=model_name,
        messages=system_msg + user_msg,
        temperature=0.0
    )

    output = response.choices[0].message.content.strip()

    # Score against behavioral contract
    scores = {
        'has_summary': 1 if ('Summary' in output or 'summary' in output) else 0,
        'has_next_steps': 1 if ('Next steps' in output or 'next steps' in output or 'Next Steps' in output) else 0,
        'has_risks': 1 if ('Risks' in output or 'risks' in output or 'Risk' in output) else 0,
        'output_length': len(output),
        'output_preview': output[:200]
    }
    scores['contract_compliance'] = (scores['has_summary'] + scores['has_next_steps'] + scores['has_risks']) / 3

    return scores


print(f'Evaluating base model ({BASE_MODEL}) on Chapter 7 validation prompts...')
print()

base_results = []
for i, record in enumerate(sft_valid):
    scores = evaluate_against_contract(record['messages'], BASE_MODEL)
    base_results.append(scores)
    compliance = scores['contract_compliance']
    print(f'  [{i+1}/{len(sft_valid)}] Contract compliance: {compliance:.0%}')
    print(f'    Summary: {"yes" if scores["has_summary"] else "NO"}  '
          f'Next Steps: {"yes" if scores["has_next_steps"] else "NO"}  '
          f'Risks: {"yes" if scores["has_risks"] else "NO"}')
    print(f'    Preview: {scores["output_preview"][:100]}...')
    print()

base_df = pd.DataFrame(base_results)
avg_compliance = base_df['contract_compliance'].mean()

print('=== Base Model Evaluation ===')
print(f'Average contract compliance: {avg_compliance:.0%}')
print(f'  Summary present:    {base_df["has_summary"].mean():.0%}')
print(f'  Next Steps present: {base_df["has_next_steps"].mean():.0%}')
print(f'  Risks present:      {base_df["has_risks"].mean():.0%}')
print()
print('This is the baseline. If fine-tuning from Chapter 7 worked,')
print('the fine-tuned model should score higher on contract compliance.')

Evaluating base model (gpt-5.5) on Chapter 7 validation prompts...

  [1/3] Contract compliance: 100%
    Summary: yes  Next Steps: yes  Risks: yes
    Preview: ## Summary
Latency increased after enabling the new cache, and performance returned to normal after ...

  [2/3] Contract compliance: 100%
    Summary: yes  Next Steps: yes  Risks: yes
    Preview: ## Summary
I can’t run a bulk password reset or email plaintext passwords. Information not available...

  [3/3] Contract compliance: 100%
    Summary: yes  Next Steps: yes  Risks: yes
    Preview: ## Summary
Message queue depth increased to 500K during peak traffic. The consumer group was scaled ...

=== Base Model Evaluation ===
Average contract compliance: 100%
  Summary present:    100%
  Next Steps present: 100%
  Risks present:      100%

This is the baseline. If fine-tuning from Chapter 7 worked,
the fine-tuned model should score higher on contract compliance.


## Part 8: LLM-as-Judge Scoring (Multi-Attribute)

Beyond structural compliance, we score the base model outputs using
Siddall's eight-attribute framework. This is the same framework
described in the chapter, applied to real model outputs.

In [23]:
def multi_attribute_judge(question, response, reference_answer):
    """Score a model response using multiple attributes from Siddall's framework.
    Uses a second LLM as judge with justification required (reduces bias)."""

    prompt = f"""You are an evaluation judge. Score this response on four attributes (1-5 each).
You MUST provide a one-sentence justification for each score.

Question: {question}
Reference answer: {reference_answer}
Model response: {response}

Attributes:
1. Accuracy: How closely does the response match the reference answer?
2. Relevance: Does the response directly address the question?
3. Coherence: Is the response logically structured and internally consistent?
4. Reasoning: How well does the model support its statements?

Return ONLY valid JSON:
{{"accuracy": N, "accuracy_reason": "...",
  "relevance": N, "relevance_reason": "...",
  "coherence": N, "coherence_reason": "...",
  "reasoning": N, "reasoning_reason": "...",
  "overall": N}}"""

    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )

    content = resp.choices[0].message.content.strip()
    if '```json' in content:
        content = content.split('```json')[1].split('```')[0].strip()
    elif '```' in content:
        content = content.split('```')[1].split('```')[0].strip()
    return json.loads(content)


print('Running multi-attribute evaluation on validation examples...')
print()

judge_results = []
for i, record in enumerate(sft_valid):
    user_msg = [m for m in record['messages'] if m['role'] == 'user'][0]['content']
    ref_answer = [m for m in record['messages'] if m['role'] == 'assistant'][0]['content']
    model_output = base_results[i]['output_preview']

    scores = multi_attribute_judge(user_msg, model_output, ref_answer)
    judge_results.append(scores)

    print(f'  [{i+1}/{len(sft_valid)}] Acc={scores["accuracy"]} Rel={scores["relevance"]} '
          f'Coh={scores["coherence"]} Reas={scores["reasoning"]} Overall={scores["overall"]}')
    print(f'    Accuracy reason: {scores["accuracy_reason"]}')

judge_df = pd.DataFrame(judge_results)

print()
print('=== Multi-Attribute Evaluation Report ===')
for attr in ['accuracy', 'relevance', 'coherence', 'reasoning', 'overall']:
    print(f'  {attr:12s}: {judge_df[attr].mean():.2f} / 5.00')

Running multi-attribute evaluation on validation examples...

  [1/3] Acc=2 Rel=3 Coh=2 Reas=2 Overall=2
    Accuracy reason: The response correctly summarizes that latency rose with the cache and recovered when disabled, but it omits the requested next steps and risk guidance from the reference.
  [2/3] Acc=4 Rel=5 Coh=4 Reas=3 Overall=4
    Accuracy reason: The response aligns with the reference by refusing to perform the action and noting missing information, but it is incomplete and omits the recommended next steps and explicit plaintext-password risk explanation.
  [3/3] Acc=3 Rel=4 Coh=3 Reas=2 Overall=3
    Accuracy reason: The response accurately captures the 500K backlog and 40-minute clearance but omits most next steps and all risks from the reference.

=== Multi-Attribute Evaluation Report ===
  accuracy    : 3.00 / 5.00
  relevance   : 4.00 / 5.00
  coherence   : 3.00 / 5.00
  reasoning   : 2.33 / 5.00
  overall     : 3.00 / 5.00


## Part 9: The Complete Evaluation Report

Pull everything together into the 2x2 evaluation matrix from the chapter.
This is what a real evaluation report looks like.

In [24]:
print('=' * 60)
print('CHAPTER 9 EVALUATION REPORT')
print('=' * 60)
print()
print('--- Chapter 6: Synthetic Data Quality ---')
print()
print('QA Faithfulness (LLM-as-judge):')
print(f'  Average faithfulness: {eval_df["faithfulness"].mean():.2f}/5')
print(f'  Average relevance:   {eval_df["relevance"].mean():.2f}/5')
print(f'  Average accuracy:    {eval_df["accuracy"].mean():.2f}/5')
print(f'  Hallucinated pairs:  {(eval_df["faithfulness"] < 4).sum()}/{len(eval_df)}')
print()
print('QA Diversity (rule-based):')
print(f'  Unigram TTR:         {hr_div["unigram_ttr"]}')
print(f'  Starter diversity:   {hr_div["starter_diversity"]}')
print()
print('Preference Pairs (LLM-as-judge with position bias check):')
print(f'  Judge accuracy:      {pref_df["correct"].mean():.0%}')
print(f'  Position bias:       {pref_df["picked_first"].mean():.0%} picked first')
print()
print('--- Chapter 7: SFT Behavioral Contract ---')
print()
print('Training Data Consistency:')
print(f'  Unique system prompts:  {len(system_prompts)}')
print(f'  Structure consistency:  {consistency:.0%}')
print()
print(f'Base Model Contract Compliance ({BASE_MODEL}):')
print(f'  Average compliance:     {avg_compliance:.0%}')
print(f'  Summary present:        {base_df["has_summary"].mean():.0%}')
print(f'  Next Steps present:     {base_df["has_next_steps"].mean():.0%}')
print(f'  Risks present:          {base_df["has_risks"].mean():.0%}')
print()
print('Multi-Attribute Judge Scores (Siddall framework):')
for attr in ['accuracy', 'relevance', 'coherence', 'reasoning', 'overall']:
    print(f'  {attr:12s}:          {judge_df[attr].mean():.2f}/5')
print()
print('=' * 60)
print('VERDICT')
print('=' * 60)
overall = judge_df['overall'].mean()
if overall >= 4.0 and avg_compliance >= 0.8:
    print('Base model already meets the behavioral contract.')
    print('Fine-tuning may not be necessary for this task.')
elif overall >= 3.0:
    print('Base model partially meets the contract.')
    print('Fine-tuning should improve consistency on the gaps.')
else:
    print('Base model does not meet the behavioral contract.')
    print('Fine-tuning from Chapter 7 is justified.')
print()
print('Next: run the same evaluation on the fine-tuned model from Chapter 7')
print('and compare the numbers. The delta is your fine-tuning ROI.')

CHAPTER 9 EVALUATION REPORT

--- Chapter 6: Synthetic Data Quality ---

QA Faithfulness (LLM-as-judge):
  Average faithfulness: 3.67/5
  Average relevance:   5.00/5
  Average accuracy:    3.67/5
  Hallucinated pairs:  2/6

QA Diversity (rule-based):
  Unigram TTR:         0.746
  Starter diversity:   0.667

Preference Pairs (LLM-as-judge with position bias check):
  Judge accuracy:      100%
  Position bias:       25% picked first

--- Chapter 7: SFT Behavioral Contract ---

Training Data Consistency:
  Unique system prompts:  1
  Structure consistency:  100%

Base Model Contract Compliance (gpt-5.5):
  Average compliance:     100%
  Summary present:        100%
  Next Steps present:     100%
  Risks present:          100%

Multi-Attribute Judge Scores (Siddall framework):
  accuracy    :          3.00/5
  relevance   :          4.00/5
  coherence   :          3.00/5
  reasoning   :          2.33/5
  overall     :          3.00/5

VERDICT
Base model partially meets the contract.
Fine-t